# 01 — Exploration du jeu de données Mobilisator

Ce notebook charge les données électorales et dresse un premier portrait statistique des 9 989 communes françaises incluses dans le dataset.

In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.titlesize'] = 14
DATA_FILE = Path('../../public/cities/cities-data.json')

In [ ]:
with open(DATA_FILE) as f:
    raw = json.load(f)

cities = list(raw.values())
print(f"{len(cities)} communes chargées")
print("\nStructure d'une commune :")
print(json.dumps({k: v for k, v in cities[0].items() if k not in ('Tour 1', 'Tour 2', 'population', 'Analyse')}, indent=2))

In [ ]:
rows = []
for c in cities:
    t1 = c.get('Tour 1', {})
    t2 = c.get('Tour 2')
    analyse = c.get('Analyse', {})
    pop = c.get('population', {})

    row = {
        'id': c['id'],
        'nom': c['nom_standard'],
        'slug': c['slug'],
        'code_dept': c['code_departement'],
        'dept': c['libelle_departement'],
        # Tour 1
        't1_inscrits': t1.get('Inscrits'),
        't1_abstentions': t1.get('Abstentions'),
        't1_pct_abs': t1.get('% Abs/Ins'),
        't1_votants': t1.get('Votants'),
        't1_exprimes': t1.get('Exprimés'),
        't1_blancs': t1.get('Blancs'),
        't1_nuls': t1.get('Nuls'),
        't1_nb_listes': len(t1.get('resultats', [])),
        # Tour 2
        'a_tour2': t2 is not None,
        't2_pct_abs': t2.get('% Abs/Ins') if t2 else None,
        # Analyse
        'votes_decisifs': analyse.get('Votes décisifs'),
        'tour_decisif': analyse.get('tour décisif'),
        'majeurs': analyse.get('majeurs'),
        'non_votants_1839': analyse.get('Non votants de 18-39'),
        'pop_1839': analyse.get('Pop 18-39'),
        'pop_18plus': analyse.get('Pop 18+'),
        'non_votants': analyse.get('Non votants'),
        'part_ne_votant_pas': analyse.get('Part ne votant pas'),
    }

    # Population totale
    if pop:
        row['pop_totale'] = sum(pop.values())
    else:
        row['pop_totale'] = None

    rows.append(row)

df = pd.DataFrame(rows)
print(df.shape)
df.head(3)

## Statistiques descriptives

In [ ]:
df[['t1_inscrits', 't1_pct_abs', 't1_nb_listes', 'pop_totale', 'part_ne_votant_pas']].describe().round(2)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Distribution du taux d'abstention T1
axes[0].hist(df['t1_pct_abs'].dropna(), bins=50, edgecolor='white', color='steelblue')
axes[0].set_title("Distribution de l'abstention (Tour 1)")
axes[0].set_xlabel("% Abstention")
axes[0].set_ylabel("Nombre de communes")

# Distribution de la population
df_pop = df['pop_totale'].dropna()
axes[1].hist(df_pop[df_pop < 50000], bins=60, edgecolor='white', color='coral')
axes[1].set_title("Population des communes (< 50 000 hab.)")
axes[1].set_xlabel("Population")

# Nombre de listes T1
df['t1_nb_listes'].value_counts().sort_index().plot(kind='bar', ax=axes[2], color='mediumseagreen', edgecolor='white')
axes[2].set_title("Nombre de listes au Tour 1")
axes[2].set_xlabel("Nb listes")
axes[2].set_ylabel("Nb communes")

plt.tight_layout()
plt.savefig('../outputs/01_exploration.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
print(f"Communes avec Tour 2 : {df['a_tour2'].sum()} ({df['a_tour2'].mean()*100:.1f}%)")
print(f"Communes sans données population : {df['pop_totale'].isna().sum()}")
print(f"\nTop 10 communes par nombre d'inscrits :")
df.nlargest(10, 't1_inscrits')[['nom', 'dept', 't1_inscrits', 't1_pct_abs']].to_string(index=False)